In [0]:
select
   tdst.store_id
  ,tdst.store_name
  ,tdst.store_group
  ,count(distinct tfs.invoice_id)                                                  as sales_quantity
  ,round(sum(tfs.part_quantity * tfs.part_unit_price), 2)                          as billing
  ,round(sum(tfs.part_quantity * (tfs.part_unit_price - tfs.part_unit_cost)), 2)   as profit
  ,round(sum(tfs.part_quantity), 2)                                                as product_quantity
from workspace.pj_sales.tb_fact_sale_gold tfs
inner join workspace.pj_sales.tb_dim_store_gold tdst
  on tdst._sk_store = tfs._sk_store
group by
   tdst.store_id
  ,tdst.store_name
  ,tdst.store_group

In [0]:
with
  aggregated_store as (
    select
       tdst.store_id
      ,tdst.store_name
      ,tdst.store_group
      ,count(distinct tfs.invoice_id)                                                  as sales_quantity
      ,round(sum(tfs.part_quantity * tfs.part_unit_price), 2)                          as billing
      ,round(sum(tfs.part_quantity * (tfs.part_unit_price - tfs.part_unit_cost)), 2)   as profit
      ,round(sum(tfs.part_quantity), 2)                                                as product_quantity
    from workspace.pj_sales.tb_fact_sale_gold tfs
    inner join workspace.pj_sales.tb_dim_store_gold tdst
      on tdst._sk_store = tfs._sk_store
    group by
       tdst.store_id
      ,tdst.store_name
      ,tdst.store_group
  )
select
  ast.store_id
  ,ast.store_name
  ,ast.store_group
  ,ast.sales_quantity
  ,ast.billing
  ,ast.profit
  ,ast.product_quantity
  ,dense_rank() over (order by ast.billing   desc)                          as rank_by_billing
from aggregated_store ast

In [0]:
create or replace view workspace.pj_sales.vw_rank_by_store as (
  with
    aggregated_store as (
      select
         tdst.store_id
        ,tdst.store_name
        ,tdst.store_group
        ,count(distinct tfs.invoice_id)                                                  as sales_quantity
        ,round(sum(tfs.part_quantity * tfs.part_unit_price), 2)                          as billing
        ,round(sum(tfs.part_quantity * (tfs.part_unit_price - tfs.part_unit_cost)), 2)   as profit
        ,round(sum(tfs.part_quantity), 2)                                                as product_quantity
      from workspace.pj_sales.tb_fact_sale_gold tfs
      inner join workspace.pj_sales.tb_dim_store_gold tdst
        on tdst._sk_store = tfs._sk_store
      where tfs.part_quantity > 0
      and   tfs.part_unit_price > 0
      and   tfs.part_unit_cost > 0
      group by
         tdst.store_id
        ,tdst.store_name
        ,tdst.store_group
    )
  select
    ast.store_id
    ,ast.store_name
    ,ast.store_group
    ,ast.sales_quantity
    ,ast.billing
    ,ast.profit
    ,ast.product_quantity
    ,dense_rank() over (order by ast.billing   desc)                          as rank_by_billing
  from aggregated_store ast
)